<a href="https://colab.research.google.com/github/lunatrifx/LocalESMFold/blob/main/LocalESMFold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch accelerate
# install transformers, torch, accelerate libraries all in one line


In [ ]:
from transformers import EsmForProteinFolding, AutoTokenizer
import torch
import os

# Locate and specify the model
tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)

from google.colab import userdata
userdata.get('HF_TOKEN')
model = model.cuda() # use CUDA
model.esm = model.esm.half() # save VRAM


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

In [ ]:
protein_name = input("Please enter the protein name (this will be used in naming the output file): ")
sequence = input("Enter the amino acid sequence: ")

In [ ]:
tokenized = tokenizer([sequence], return_tensors="pt", add_special_tokens=False)
tokenized = {k: v.cuda() for k, v in tokenized.items()}

In [ ]:
with torch.no_grad():
  output = model(**tokenized)

In [ ]:
# PDB output
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37

pdb_string = model.output_to_pdb(output)[0]
with open(f"{protein_name}.pbd", "w") as f:
  f.write(pdb_string)

  print("Operation complete. Download dopamine_d2.pdb, and open it in ChimeraX or PyMol to continue analysis.")

In place of Chimera X, which is Desktop only, this demo will use py3Dmol.

In [ ]:
!pip install py3Dmol

In [ ]:
import py3Dmol

with open(f"{protein_name}", "r") as f:
  pdb_data = f.read()

view = py3Dmol.view(width=1920, height=1080)
view.addModel(pdb_data, "pdb")
# color by confidence or pLDDT score
view.setStyle:({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90 }}})
view.zoomTo()
view.show()

In [3]:
# This isn't part of the code, but I just wanted to add this since I had to do multiple revisions
# due to an error when importing this to Github (rendering Notebook issue)

import json

notebook_path = "/content/LocalESMFold.ipynb"

with open(notebook_path, "r") as f:
  nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
  del nb["metadata"]["widgets"]

with open(notebook_path, "w") as f:
  json.dump(nb, f, indent=1)

print("Done. Widget metadata removed. Push to Github.")

##### Credit to zabop and Julio Cezar Silva on stackoverflow for the initial idea of using bash to compile a html fix. #####

Done. Widget metadata removed. Push to Github.
